# MODELO NEURALPROPHET PARA EL CONTAMINANTE CO PARA MADRID

En este notebook, vamos a ajustar modelos NeuralProphet para el contaminante CO sobre la ciudad de Madrid.

NeuralProphet surge como una herramienta potente y accesible para abordar la predicción de series temporales mediante redes neuronales. Es una librería de código abierto desarrollada por Facebook, que combina la simplicidad de uso con la capacidad de modelar y predecir series temporales de manera efectiva. Se basa en una arquitectura de redes neuronales y aprovecha las capacidades del aprendizaje profundo para capturar patrones complejos y realizar predicciones precisas.

Una de las ventajas distintivas de NeuralProphet es su facilidad de uso. A diferencia de otras librerías de *Machine Learning*, NeuralProphet está diseñada para ser accesible incluso para quienes no tienen experiencia en ciencia de datos.

Además, permite modelar tendencias no lineales, estacionalidad, efectos de eventos y cambios en la tendencia a lo largo del tiempo. También incluye regresiones externas, lo que permite añadir variables adicionales que puedan influir en los datos de la serie temporal.

Otra característica de NeuralProphet es su capacidad para manejar automáticamente múltiples hiperparámetros del modelo, como la selección de la arquitectura de la red neuronal, la regularización y la optimización del aprendizaje. Esto reduce la carga de trabajo de los usuarios y simplifica el proceso de ajuste del modelo.

NeuralProphet consta de un modelo aditivo en el que cada serie temporal se modela como la combinación de 6 componentes. Matemáticamente, esto se puede expresar como:

$$\hat{y}_t = T(t) + S(t) + E(t) + F(t) + A(t) + L(t),$$

donde: 

* $T(t)$ es la componente de tendencia.
* $S(t)$ es la componente estacional.
* $E(t)$ representa los efectos de los días festivos.
* $F(t)$ son los efectos de regresión para variables exógenas conocidas en el futuro.
* $A(t)$ son los efectos de autorregresión.
* $L(t)$ son los efectos de regresión para observaciones retardadas de variables exógenas.

### Componente de tendencia

$$T(t) = ( \delta_0 + a(t)^T\delta)t + (\rho_0 + a(t)^T\rho),$$

donde:

* $\delta$ es un vector de ajustes de la tasa de crecimiento.
* $\rho$ es un vector de ajustes de desplazamiento.
* $\delta_0$ es la tasa de crecimiento inicial.
* $\rho_0$ es el desplazamiento inicial.
* $a(t)$ es un vector binario que representa si el tiempo $t$ ha pasado cada uno de los puntos de cambio.

### Componente estacional

$$S(t) = \sum_{p \in P} S_p^*(t),$$

con 

$$
S^*_p(t) =
\begin{cases}
T(t) \cdot S_p(t), & \text{si } S_p \text{ es multiplicativa} \\
S_p(t), & \text{en otro caso}
\end{cases}
$$

y

$$
S_p(t) = \sum_{n=1}^{N} \left(a_n cos\left(\frac{2 \pi t n}{p}\right) + b_n sin\left(\frac{2 \pi t n}{p}\right)\right),
$$

donde:

* $P$ es el conjunto de todas las estacionalidades.
* $N$ es el número de términos en la serie de Fourier.

### Efectos de días festivos

Incorporamos una lista con los días festivos en el modelo, asumiendo que sus efectos son independientes.

### Regresores retardados

$$
L(t) = \sum_{x \in X} L_x(x_{t-1}, x_{t-2}, \dots, x_{t-p}),
$$

donde:

* $X$ es un conjunto de $m$ covariables de longitud $T$.

### Regresores futuros

$$
F(t) = \sum_{f \in \mathcal{F}} F^*_f(t),
$$

con

$$
F^*_f(t) =
\begin{cases}
T(t) \cdot d_f f(t), & \text{si } f \text{ es multiplicativa} \\
d_f f(t), & \text{en otro caso}
\end{cases},
$$

donde:

* $F$ es el conjunto de regresiones futuras.
* $f(t)$ es el valor de la variable futura $f$ en el tiempo $t$.
* $d_f$ es el coeficiente del modelo correspondiente al retorno futuro $f$.


Importamos las librerías y definimos las rutas.

In [ ]:
import numpy as np
import pandas as pd
import random
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn import metrics
import itertools
import random
import torch
from neuralprophet import NeuralProphet
from pylab import rcParams
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterSampler
from neuralprophet import set_log_level
from sklearn.model_selection import ParameterGrid

import warnings 
warnings.filterwarnings("ignore")
warnings.filterwarnings(action="ignore",category=FutureWarning)

plt.style.use("fivethirtyeight")
light_style = {
    "figure.facecolor": "#d9effb",   
    "axes.facecolor": "#d9effb",
    "savefig.facecolor": "#d9effb",
    "axes.grid": True,
    "axes.grid.which": "both",
    "axes.spines.left": True,
    "axes.spines.right": True,
    "axes.spines.top": True,
    "axes.spines.bottom": True,
    "grid.color": "#a9d3f2",
    "grid.linewidth": "0.8",
    "text.color": "#333333",
    "axes.labelcolor": "#333333",
    "axes.labelweight": "black",  
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "font.size": 12,
    "axes.titleweight": "bold",  
    "legend.fontsize": 12,
    "legend.title_fontsize": 12,
}
plt.rcParams.update(light_style)
rcParams['figure.figsize'] = (18, 7)

# Reducimos los mensajes generados durante el entrenamiento
set_log_level("ERROR")


import sys
import importlib
from pathlib import Path

SCRIPTS_PATH = Path.cwd().parents[2]

if str(SCRIPTS_PATH) not in sys.path:
    sys.path.append(str(SCRIPTS_PATH))
    
import utils
importlib.reload(utils)
from utils import EVALUAR_METRICAS, ENTRENAR_EVALUAR_NEURALPROPHET

In [48]:
BASE_PATH = Path("..", "..", "..", "..")
FOLDER_DATA = BASE_PATH / "datasets" / "eda_archivos_cont_clima_indices"

## Carga de los datos y división del conjunto de datos

Cargamos los datos.

In [49]:
df = pd.read_csv(FOLDER_DATA / "dataset_cont_clima_indices_limpio.csv")

Filtramos las columnas que realmente necesitamos y como ciudad elegimos únicamente Madrid. Dado que todo el trabajo exploratorio ya se encuentra realizado en el *notebook* para el modelo Prophet, nos quedamos únicamente con las variables exógenas definidas anteriormente.

In [50]:
# Listado de columnas seleccionadas
columnas = [
    'Start', 'CO (mg.m-3)', 'city', 'temperature_2m', 'snowfall',
    'relative_humidity_2m', 'precipitation', 'rain',
    'surface_pressure', 'cloudcover', 'windspeed_10m', 
    'shortwave_radiation', 'boundary_layer_height', 'NDVI', 'NDBI', 'Año'
]

# Filtrar por Madrid y seleccionar las columnas
df1 = df[df['city'] == 'Madrid'][columnas].copy()

# Resetear los índices para que empiecen desde 0
df1.reset_index(drop=True, inplace=True)

# Eliminamos la columna 'city'
df1.drop(columns=['city'], inplace=True)

In [51]:
# ==============================================================================
# División cronológica del conjunto de datos
# ==============================================================================

train = df1[df1["Año"] <= 2020].copy()

validation = df1[
    (df1["Año"] >= 2021) &
    (df1["Año"] <= 2022)
].copy()

test = df1[df1["Año"] >= 2023].copy()

print(f"Entrenamiento: {train['Start'].min()} -> {train['Start'].max()}")
print(f"Validación:    {validation['Start'].min()} -> {validation['Start'].max()}")
print(f"Prueba:        {test['Start'].min()} -> {test['Start'].max()}")

print()
print(f"Nº muestras entrenamiento: {len(train):,}")
print(f"Nº muestras validación:    {len(validation):,}")
print(f"Nº muestras prueba:        {len(test):,}")

Entrenamiento: 2013-01-01 00:00:00 -> 2020-12-31 23:00:00
Validación:    2021-01-01 00:00:00 -> 2022-12-31 23:00:00
Prueba:        2023-01-01 00:00:00 -> 2024-12-31 23:00:00

Nº muestras entrenamiento: 70,128
Nº muestras validación:    17,520
Nº muestras prueba:        17,544


NeuralProphet espera que los datos estén formateados de una manera específica. El modelo requiere una columna *ds* que contenga las fechas y una columna *y* que contenga los valores que queremos modelar/predecir.

In [52]:
train = train.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})
validation = validation.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})
test = test.rename(columns={"Start": "ds", "CO (mg.m-3)": "y"})

In [53]:
variables_exogenas= [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "surface_pressure",
    "cloudcover",
    "windspeed_10m",
    "shortwave_radiation",
    "boundary_layer_height",
    "NDVI",
    "NDBI"
]

Nos quedamos exclusivamente con las variables necesarias para el modelado.

In [54]:
# Lista de columnas que quieres mantener
columnas = ['ds', 'y'] + variables_exogenas

train = train[columnas]
validation = validation[columnas]

Definimos los regresores retardados y los regresores futuros.

In [55]:
regresores_retardados = variables_exogenas.copy()
regresores_retardados.remove("NDBI")
regresores_retardados.remove("NDVI")

regresores_futuros = ["NDVI", "NDBI"]


## Selección de los mejores hiperparámetros

Una vez definidos los conjuntos de entrenamiento, validación y prueba, continuamos seleccionando la configuración más adecuada del modelo **NeuralProphet**. Para ello, ajustamos cada modelo utilizando exclusivamente las observaciones del conjunto de entrenamiento, correspondientes al periodo comprendido entre 2013 y 2020, y evaluamos posteriormente su capacidad predictiva sobre el conjunto de validación, formado por los años 2021 y 2022.

La selección de los hiperparámetros se lleva a cabo mediante la comparación de diferentes configuraciones del modelo. Para cada combinación, NeuralProphet se entrena con el conjunto de entrenamiento y genera predicciones sobre las observaciones del conjunto de validación. Posteriormente, se calcula el error cuadrático medio, MSE, entre los valores reales y las predicciones obtenidas. La configuración seleccionada será aquella que presente el menor MSE sobre el conjunto de validación.

Para realizar las predicciones, se establece un horizonte de una observación mediante `n_forecasts = 1`. Dado que los datos presentan una frecuencia horaria, el modelo realiza predicciones a una hora vista a lo largo de todo el conjunto de validación. Además, para cada instante se pueden utilizar observaciones anteriores de la concentración de CO mediante la componente autorregresiva del modelo.

De acuerdo con el análisis exploratorio realizado previamente, todos los modelos incorporan simultáneamente una estacionalidad anual y una estacionalidad diaria, mientras que la estacionalidad semanal no se incluye. En ambos casos se evalúan diferentes números de términos de Fourier con el objetivo de determinar el grado de flexibilidad más adecuado para representar los patrones temporales presentes en la serie.

Asimismo, se incorporan los días festivos nacionales de España en todas las configuraciones evaluadas, con el objetivo de considerar posibles modificaciones en los patrones de movilidad y actividad asociados a estas fechas.

Además de la propia componente autorregresiva, NeuralProphet permite diferenciar entre variables exógenas conocidas únicamente en el pasado y variables cuyos valores se consideran conocidos en el futuro. Tras analizar previamente la multicolinealidad entre las variables meteorológicas y ambientales, se excluyen `rain` y `snowfall` debido a su elevada redundancia con la precipitación total.

Las siguientes variables se incorporan como **regresores retardados**, puesto que sus valores futuros no se consideran conocidos en el momento de realizar la predicción:

* Temperatura a dos metros.
* Humedad relativa a dos metros.
* Precipitación total.
* Presión en superficie.
* Nubosidad.
* Velocidad del viento a diez metros.
* Radiación solar de onda corta.
* Altura de la capa límite planetaria.

Por otro lado, el NDVI y el NDBI se incorporan como **regresores futuros**, al considerar que sus valores están disponibles para las fechas que se desean predecir:

* Índice de vegetación de diferencia normalizada, NDVI.
* Índice de edificación de diferencia normalizada, NDBI.

Los regresores retardados utilizan el mismo número de observaciones anteriores que la componente autorregresiva del modelo. De este modo, el hiperparámetro `n_lags` determina tanto la cantidad de concentraciones anteriores de CO empleadas por AR-Net como el número de observaciones previas utilizadas para las variables exógenas retardadas.

A diferencia de Prophet, NeuralProphet no incorpora un crecimiento logístico basado en una capacidad máxima y un límite inferior. Por ello, todos los modelos considerados emplean una tendencia lineal mediante `growth = "linear"`.

Los principales hiperparámetros evaluados son:

* La proporción inicial de la serie en la que se sitúan los puntos de cambio de tendencia, mediante `changepoints_range`.
* La regularización de los cambios de tendencia, mediante `trend_reg`.
* El tipo de combinación de las componentes estacionales, mediante `seasonality_mode`.
* La regularización de las componentes estacionales, mediante `seasonality_reg`.
* El número de términos de Fourier utilizados para representar las estacionalidades anual y diaria, mediante `yearly_seasonality` y `daily_seasonality`.
* El número de observaciones anteriores utilizadas por la componente autorregresiva, mediante `n_lags`.
* La arquitectura de la red autorregresiva AR-Net, mediante `ar_layers`.
* La arquitectura de la red neuronal asociada a los regresores retardados, mediante `lagged_reg_layers`.
* El número de épocas utilizadas durante el entrenamiento, mediante `epochs`.
* El tamaño de los lotes utilizados durante el descenso de gradiente, mediante `batch_size`.

Una vez evaluadas todas las configuraciones, se selecciona aquella que presenta el menor MSE sobre el conjunto de validación. Posteriormente, el modelo correspondiente a dicha configuración se vuelve a entrenar sobre el conjunto de entrenamiento y se calculan el resto de métricas de evaluación sobre el conjunto de validación.


In [ ]:
# ==============================================================================
# Valores candidatos de los hiperparámetros
# ==============================================================================

param_grid = {

    # --------------------------------------------------------------------------
    # Tendencia
    # --------------------------------------------------------------------------

    # Parte inicial de la serie en la que se colocan los puntos de cambio
    "changepoints_range": [
        0.8,
        0.9
    ],

    # Regularización de los cambios de tendencia
    "trend_reg": [
        0.1,
        1.0
    ],


    # --------------------------------------------------------------------------
    # Estacionalidad
    # --------------------------------------------------------------------------

    # Tipo de estacionalidad
    "seasonality_mode": [
        "multiplicative"
    ],

    # Regularización de las componentes estacionales
    "seasonality_reg": [
        0.1,
        1.0
    ],

    # Número de términos de Fourier de la estacionalidad anual
    "yearly_seasonality": [
        10,
        20
    ],

    # Número de términos de Fourier de la estacionalidad diaria
    "daily_seasonality": [
        10,
        20
    ],

    # --------------------------------------------------------------------------
    # Autorregresión
    # --------------------------------------------------------------------------

    # Número de valores anteriores de CO utilizados por AR-Net
    "n_lags": [
        48,
        168
    ],

    # Capas ocultas de la red autorregresiva
    "ar_layers": [
        [],
        [32],
        [64, 32]
    ],

    # Capas ocultas para los regresores retardados
    "lagged_reg_layers": [
        [],
        [32],
        [64, 32]
    ],

    # --------------------------------------------------------------------------
    # Entrenamiento
    # --------------------------------------------------------------------------

    # Número de épocas
    "epochs": [
        50,
        100
    ],

    # Tamaño de los lotes
    "batch_size": [
        64,
        128,
        256
    ],

    # --------------------------------------------------------------------------
    # Festivos
    # --------------------------------------------------------------------------

    # Incorporación de los festivos nacionales de España
    "usar_festivos": [
        True
    ],
}

In [62]:
# ==============================================================================
# Generación de todas las combinaciones posibles
# ==============================================================================

configuraciones = list(
    ParameterGrid(param_grid)
)

print(
    f"Número total de configuraciones que se evaluarán: "
    f"{len(configuraciones)}"
)

Número total de configuraciones que se evaluarán: 3456


In [ ]:
# ==============================================================================
# Ajuste y evaluación de las diferentes configuraciones
# ==============================================================================

resultados_busqueda = []

for numero_configuracion, params in enumerate(configuraciones, start=1):

    # Extraemos el indicador de utilización de festivos, ya que no es
    # un hiperparámetro que se pase directamente al constructor
    usar_festivos = params["usar_festivos"]

    # Seleccionamos únicamente los parámetros admitidos por NeuralProphet
    parametros_modelo = {

        # Tendencia
        "growth": "linear",
        "changepoints_range": params["changepoints_range"],
        "trend_reg": params["trend_reg"],

        # Estacionalidad
        "seasonality_mode": params["seasonality_mode"],
        "seasonality_reg": params["seasonality_reg"],
        "yearly_seasonality": params["yearly_seasonality"],
        "weekly_seasonality": params["weekly_seasonality"],
        "daily_seasonality": params["daily_seasonality"],

        # Autorregresión
        "n_lags": params["n_lags"],
        "n_forecasts": 1,
        "ar_layers": params["ar_layers"],
        "lagged_reg_layers": params["lagged_reg_layers"],

        # Entrenamiento
        "epochs": params["epochs"],
        "batch_size": params["batch_size"]
    }

    try:

        # ----------------------------------------------------------------------
        # Fijación de las semillas
        # ----------------------------------------------------------------------
        np.random.seed(42)
        random.seed(42)
        torch.manual_seed(42)

        if torch.cuda.is_available():

            torch.cuda.manual_seed_all(42)

        # ----------------------------------------------------------------------
        # Copia de los conjuntos de entrenamiento y validación
        # ----------------------------------------------------------------------
        datos_entrenamiento = train.copy()
        datos_validacion = validation.copy()

        # ----------------------------------------------------------------------
        # Conversión de las fechas
        # ----------------------------------------------------------------------
        datos_entrenamiento["ds"] = pd.to_datetime(
            datos_entrenamiento["ds"]
        )

        datos_validacion["ds"] = pd.to_datetime(
            datos_validacion["ds"]
        )

        # ----------------------------------------------------------------------
        # Ordenación cronológica
        # ----------------------------------------------------------------------
        datos_entrenamiento = datos_entrenamiento.sort_values(
            by="ds"
        ).reset_index(drop=True)

        datos_validacion = datos_validacion.sort_values(
            by="ds"
        ).reset_index(drop=True)

        # ----------------------------------------------------------------------
        # Creación del modelo
        # ----------------------------------------------------------------------
        modelo = NeuralProphet(
            **parametros_modelo
        )

        # ----------------------------------------------------------------------
        # Incorporación de los regresores retardados
        # ----------------------------------------------------------------------
        for variable in regresores_retardados:

            modelo.add_lagged_regressor(
                names=variable,
                n_lags=params["n_lags"],
                normalize="standardize"
            )

        # ----------------------------------------------------------------------
        # Incorporación de los regresores futuros
        # ----------------------------------------------------------------------
        for variable in regresores_futuros:

            modelo.add_future_regressor(
                name=variable,
                normalize="standardize",
                mode=params["seasonality_mode"]
            )

        # ----------------------------------------------------------------------
        # Incorporación de los festivos nacionales de España
        # ----------------------------------------------------------------------
        if usar_festivos:

            modelo.add_country_holidays(
                country_name="ES"
            )

        # ----------------------------------------------------------------------
        # Entrenamiento del modelo
        # ----------------------------------------------------------------------
        modelo.fit(
            datos_entrenamiento,
            freq="h",
            progress=None
        )

        # ----------------------------------------------------------------------
        # Preparación de los datos para la predicción
        # ----------------------------------------------------------------------
        # NeuralProphet necesita observaciones anteriores al comienzo de la
        # validación para construir los primeros rezagos. Por ello, se añaden
        # las últimas n_lags observaciones del conjunto de entrenamiento.
        contexto_entrenamiento = datos_entrenamiento.tail(
            params["n_lags"]
        ).copy()

        datos_prediccion = pd.concat(
            [
                contexto_entrenamiento,
                datos_validacion
            ],
            ignore_index=True
        )

        # Ordenamos nuevamente después de concatenar
        datos_prediccion = datos_prediccion.sort_values(
            by="ds"
        ).reset_index(drop=True)

        # ----------------------------------------------------------------------
        # Predicción sobre el conjunto de validación
        # ----------------------------------------------------------------------
        prediccion = modelo.predict(
            datos_prediccion
        )

        # ----------------------------------------------------------------------
        # Selección de las predicciones del periodo de validación
        # ----------------------------------------------------------------------
        prediccion_validacion = prediccion[
            prediccion["ds"].isin(
                datos_validacion["ds"]
            )
        ][
            [
                "ds",
                "yhat1"
            ]
        ].copy()

        # ----------------------------------------------------------------------
        # Unión de los valores reales y las predicciones
        # ----------------------------------------------------------------------
        comparacion_validacion = datos_validacion[
            [
                "ds",
                "y"
            ]
        ].merge(
            prediccion_validacion,
            on="ds",
            how="inner"
        )

        # Eliminamos las filas sin valor real o sin predicción
        comparacion_validacion = comparacion_validacion.dropna(
            subset=[
                "y",
                "yhat1"
            ]
        ).reset_index(drop=True)

        # Comprobamos que NeuralProphet haya generado predicciones válidas
        if comparacion_validacion.empty:

            raise ValueError(
                "NeuralProphet no ha generado predicciones válidas "
                "para el conjunto de validación."
            )

        # ----------------------------------------------------------------------
        # Cálculo del error cuadrático medio
        # ----------------------------------------------------------------------
        mse = mean_squared_error(
            comparacion_validacion["y"],
            comparacion_validacion["yhat1"]
        )

        # ----------------------------------------------------------------------
        # Almacenamiento del resultado
        # ----------------------------------------------------------------------
        resultado = params.copy()

        resultado["MSE_validacion"] = mse
        resultado["numero_predicciones"] = len(
            comparacion_validacion
        )
        resultado["estado"] = "Correcto"

        resultados_busqueda.append(resultado)

    except Exception as error:

        # ----------------------------------------------------------------------
        # Almacenamiento del error
        # ----------------------------------------------------------------------
        resultado = params.copy()

        resultado["MSE_validacion"] = np.nan
        resultado["numero_predicciones"] = 0
        resultado["estado"] = str(error)

        resultados_busqueda.append(resultado)

        print(
            f"Error en la configuración "
            f"{numero_configuracion}:"
        )

        print(error)


# ==============================================================================
# Conversión de los resultados en un DataFrame
# ==============================================================================

resultados_df = pd.DataFrame(
    resultados_busqueda
)


# ==============================================================================
# Selección de las configuraciones correctas
# ==============================================================================

resultados_correctos = resultados_df[
    resultados_df["estado"] == "Correcto"
].copy()

resultados_correctos = resultados_correctos.sort_values(
    by="MSE_validacion",
    ascending=True
).reset_index(drop=True)


# ==============================================================================
# Presentación de la mejor configuración
# ==============================================================================

if resultados_correctos.empty:

    print(
        "\nNo se ha podido ajustar correctamente "
        "ninguna configuración."
    )

    print(
        "\nErrores encontrados:\n"
    )

    for indice, fila in resultados_df.iterrows():

        print(
            f"Configuración {indice + 1}: "
            f"{fila['estado']}"
        )

else:

    mejor_resultado = resultados_correctos.iloc[0]

    # --------------------------------------------------------------------------
    # Identificación de las estacionalidades utilizadas
    # --------------------------------------------------------------------------
    estacionalidades_utilizadas = []

    if mejor_resultado["yearly_seasonality"] not in [
        False,
        0,
        None
    ]:

        estacionalidades_utilizadas.append(
            "yearly"
        )

    if mejor_resultado["weekly_seasonality"] not in [
        False,
        0,
        None
    ]:

        estacionalidades_utilizadas.append(
            "weekly"
        )

    if mejor_resultado["daily_seasonality"] not in [
        False,
        0,
        None
    ]:

        estacionalidades_utilizadas.append(
            "daily"
        )

    # --------------------------------------------------------------------------
    # Presentación de las estacionalidades
    # --------------------------------------------------------------------------
    print(
        "\nEstacionalidades utilizadas:"
    )

    for estacionalidad in estacionalidades_utilizadas:

        print(
            f"- {estacionalidad}"
        )

    # --------------------------------------------------------------------------
    # Presentación de los mejores hiperparámetros
    # --------------------------------------------------------------------------
    print(
        "\nMejores parámetros:\n"
    )

    for parametro in param_grid.keys():

        print(
            f"{parametro}: "
            f"{mejor_resultado[parametro]}"
        )

    # --------------------------------------------------------------------------
    # Presentación del error de validación
    # --------------------------------------------------------------------------
    print(
        f"\nMSE de validación: "
        f"{mejor_resultado['MSE_validacion']:.8f}"
    )

Estacionalidades utilizadas:
- yearly
- weekly
- daily

Mejores parámetros:

changepoints_range: 0.9
trend_reg: 0.1
seasonality_mode: multiplicative
seasonality_reg: 1.0
yearly_seasonality: 20
daily_seasonality: 20
n_lags: 168
ar_layers: [64, 32]
lagged_reg_layers: [32]
epochs: 100
batch_size: 128
usar_festivos: True

MSE de validación: 0.015328


Así, obtenemos el mejor modelo NeuralProphet para el contaminante CO en Madrid. Veamos todas sus métricas de error sobre el conjunto de validación.

In [ ]:
# ==============================================================================
# Modelo NeuralProphet con los mejores hiperparámetros
# ==============================================================================

# Copiamos los conjuntos para no modificar los originales
train_neuralprophet_final = train.copy()
validation_neuralprophet_final = validation.copy()


# ==============================================================================
# Preparación de los datos
# ==============================================================================

# Conversión de las fechas
train_neuralprophet_final["ds"] = pd.to_datetime(
    train_neuralprophet_final["ds"]
)

validation_neuralprophet_final["ds"] = pd.to_datetime(
    validation_neuralprophet_final["ds"]
)

# Ordenación cronológica
train_neuralprophet_final = train_neuralprophet_final.sort_values(
    by="ds"
).reset_index(drop=True)

validation_neuralprophet_final = validation_neuralprophet_final.sort_values(
    by="ds"
).reset_index(drop=True)


# ==============================================================================
# Fijación de las semillas
# ==============================================================================

np.random.seed(42)
random.seed(42)
torch.manual_seed(42)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


# ==============================================================================
# Creación del modelo
# ==============================================================================

modelo_neuralprophet = NeuralProphet(

    # Tendencia
    growth="linear",
    changepoints_range=0.90,
    trend_reg=0.10,

    # Estacionalidad
    seasonality_mode="multiplicative",
    seasonality_reg=1.0,
    yearly_seasonality=20,
    weekly_seasonality=False,
    daily_seasonality=20,

    # Autorregresión
    n_lags=168,
    n_forecasts=1,
    ar_layers=[64, 32],
    lagged_reg_layers=[32],

    # Entrenamiento
    epochs=100,
    batch_size=128
)


# ==============================================================================
# Incorporación de los regresores retardados
# ==============================================================================

for variable in regresores_retardados:

    modelo_neuralprophet.add_lagged_regressor(
        names=variable,
        n_lags=168,
        normalize="standardize"
    )


# ==============================================================================
# Incorporación de los regresores futuros
# ==============================================================================

for variable in regresores_futuros:

    modelo_neuralprophet.add_future_regressor(
        name=variable,
        normalize="standardize",
        mode="multiplicative"
    )


# ==============================================================================
# Incorporación de los festivos nacionales de España
# ==============================================================================

modelo_neuralprophet.add_country_holidays(
    country_name="ES"
)


# ==============================================================================
# Entrenamiento del modelo
# ==============================================================================

historial_entrenamiento = modelo_neuralprophet.fit(
    train_neuralprophet_final,
    freq="h",
    progress=None
)


# ==============================================================================
# Preparación de los datos de validación
# ==============================================================================

# NeuralProphet necesita las últimas 168 observaciones de entrenamiento
# para construir los retardos iniciales del periodo de validación.
contexto_entrenamiento = train_neuralprophet_final.tail(
    168
).copy()

datos_prediccion_validacion = pd.concat(
    [
        contexto_entrenamiento,
        validation_neuralprophet_final
    ],
    ignore_index=True
)

datos_prediccion_validacion = datos_prediccion_validacion.sort_values(
    by="ds"
).reset_index(drop=True)


# ==============================================================================
# Predicción sobre el conjunto de validación
# ==============================================================================

prediccion_validacion_completa = modelo_neuralprophet.predict(
    datos_prediccion_validacion
)


# ==============================================================================
# Selección de las predicciones del periodo de validación
# ==============================================================================

prediccion_validacion = prediccion_validacion_completa[
    prediccion_validacion_completa["ds"].isin(
        validation_neuralprophet_final["ds"]
    )
][
    [
        "ds",
        "yhat1"
    ]
].copy()


# ==============================================================================
# Unión de valores reales y predicciones
# ==============================================================================

comparacion_validacion = validation_neuralprophet_final[
    [
        "ds",
        "y"
    ]
].merge(
    prediccion_validacion,
    on="ds",
    how="inner"
)

comparacion_validacion = comparacion_validacion.dropna(
    subset=[
        "y",
        "yhat1"
    ]
).reset_index(drop=True)


# ==============================================================================
# Evaluación del modelo
# ==============================================================================

# Número de variables exógenas utilizadas
num_parametros = (
    len(regresores_retardados)
    + len(regresores_futuros)
)

metricas_validacion = EVALUAR_METRICAS(
    y_real=comparacion_validacion["y"],
    y_predicho=comparacion_validacion["yhat1"],
    num_parametros=num_parametros
)

Resultados de la evaluación del modelo
---------------------------------------
Error absoluto medio (MAE): 0.063415
Error cuadrático medio (MSE): 0.015328
Raíz del error cuadrático medio (RMSE): 0.123806
Error porcentual absoluto medio (MAPE): 10.71 %
Raíz del error cuadrático medio normalizada (NRMSE): 36.06 %


### Error de generalización

Finalmente, dado que este es el mejor modelo para el contaminante CO en Madrid, calculamos su error de generalización, que es el error que surge de entrenar el modelo con los conjuntos de entrenamiento y validación y predecir sobre el conjunto de prueba.

In [ ]:
# Unimos train y validation en un único DataFrame
train_val = pd.concat([train, validation], ignore_index=True)

test = test[[columnas]]

In [ ]:
mejores_parametros = {
    "changepoints_range": 0.9,
    "trend_reg": 0.1,
    "seasonality_mode": "multiplicative",
    "seasonality_reg": 1.0,
    "yearly_seasonality": 20,
    "weekly_seasonality": 20,
    "n_lags": 168,
    "ar_layers": [64, 32],
    "lagged_reg_layers": [32],
    "epochs": 100,
    "batch_size": 128,
    "usar_festivos": True,
}

In [ ]:
(
    modelo_neuralprophet,
    historial_entrenamiento_val,
    prediccion_test,
    comparacion_test,
    metricas_test,
    train_val_neuralprophet_final,
    test_neuralprophet_final
) = ENTRENAR_EVALUAR_NEURALPROPHET(
    train=train_val,
    validation=test,
    regresores_futuros=regresores_futuros,
    regresores_retardados=regresores_retardados,
    mejores_parametros=mejores_parametros
)

Resultados de la evaluación del modelo
--------------------------------------
Error absoluto medio (MAE): 0.065127
Error cuadrático medio (MSE): 0.018946
Raíz del error cuadrático medio (RMSE): 0.137644
Error porcentual absoluto medio (MAPE): 11.70 %
Raíz del error cuadrático medio normalizada (NRMSE): 37.65  %
